# File QA RAG Chatbot App with ChatGPT, LangChain and Chainlit

Here we will implement an advanced RAG System with ChatGPT, LangChain and Chainlit to build a File QA UI-based chatbot with the following features:

- PDF Document Upload and Indexing
- RAG System for query analysis and response
- Result streaming capabilities (Real-time output)
- Show document sources of the answer from RAG system

## Install App and LLM dependencies

In [1]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12
!pip install langchain-community==0.3.11
!pip install chainlit==1.3.2
!pip install pyngrok==7.2.2
!pip install PyMuPDF==1.24.0
!pip install chromadb==0.6.3
!pip install pydantic==2.10.1
!pip install langchain-chroma==0.2.2

  Using cached chromadb-0.6.3-py3-none-any.whl.metadata (6.8 kB)
  Using cached build-1.2.2.post1-py3-none-any.whl.metadata (6.5 kB)
  Using cached chroma_hnswlib-0.7.6-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (252 bytes)
  Using cached posthog-3.19.1-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached onnxruntime-1.21.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (4.5 kB)
  Using cached opentelemetry_instrumentation_fastapi-0.51b0-py3-none-any.whl.metadata (2.2 kB)
  Using cached pypika-0.48.9-py2.py3-none-any.whl
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached bcrypt-4.3.0-cp39-abi3-manylinux_2_34_x86_64.whl.metadata (10 kB)
  Using cached kubernetes-32.0.1-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached mmh3-5.1.0-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (16 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1

## Load OpenAI API Credentials

Here we load it from a file so we don't explore the credentials on the internet by mistake

In [2]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [3]:
import yaml

with open('api_keys.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

In [4]:
api_creds.keys()

dict_keys(['OPENAI_API_KEY', 'GEMINI_API_KEY', 'NGORK_AUTH_TOKEN', 'DEEPSEEK_API_KEY', 'TOGETHER_API_KEY', 'TAVILY_API_KEY'])

In [5]:
import os

os.environ['OPENAI_API_KEY'] = api_creds['OPENAI_API_KEY']

## Write the app code here and store it in a py file

In [7]:
# the following line is a magic command
# that will write all the code below it to the python file app.py
# we will then deploy this app.py file on the cloud server where colab is running
# if you have your own server you can just write the code in app.py and deploy it directly
%%writefile app.py

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.callbacks.base import BaseCallbackHandler
from langchain.schema.runnable.config import RunnableConfig
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import StrOutputParser
from langchain_community.vectorstores.chroma import Chroma
from operator import itemgetter
import chainlit as cl
import chromadb
import tempfile
import os
import pandas as pd

# Takes uploaded PDFs, creates document chunks, computes embeddings
# Stores document chunks and embeddings in a Vector DB
# Returns a retriever which can look up the Vector DB
# to return documents based on user input
def configure_retriever(uploaded_files):
  # Read documents
  docs = []
  temp_dir = tempfile.TemporaryDirectory()
  for file in uploaded_files:
    temp_filepath = os.path.join(temp_dir.name, file.name)
    with open(temp_filepath, "wb") as f:
      with open(file.path, 'rb') as infile:
        f.write(infile.read())
    loader = PyMuPDFLoader(temp_filepath)
    docs.extend(loader.load())

  # Split into documents chunks
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500,
                                                 chunk_overlap=200)
  doc_chunks = text_splitter.split_documents(docs)

  # Create document embeddings and store in Vector DB
  embeddings_model = OpenAIEmbeddings()
  client = chromadb.PersistentClient(path="./chroma_db")

  # Use Chroma from LangChain
  vectordb = Chroma.from_documents(
      documents=doc_chunks,
      embedding=embeddings_model,
      client=client,
      collection_name="document_collection"
  )

  # Define retriever object
  retriever = vectordb.as_retriever(search_kwargs={"k": 3})
  return retriever

@cl.on_chat_start
# this function is called when the app starts for the first time
async def when_chat_starts():
  # Create UI element to accept PDF uploads
  uploaded_files = None
  # Wait for the user to upload a file
  while uploaded_files == None:
    uploaded_files = await cl.AskFileMessage(
      content="Please upload PDF documents to continue.",
      accept=["application/pdf"],
      max_size_mb=20, max_files=5, timeout=180
    ).send()

  msg = cl.Message(content=f"Processing files please wait...") #, disable_feedback=True
  await msg.send()
  await cl.sleep(2)
  # Create retriever object based on uploaded PDFs
  retriever = configure_retriever(uploaded_files)
  msg = cl.Message(content=f"Processing completed. You can now ask questions!") #, disable_feedback=True
  await msg.send()

  # Load a connection to ChatGPT LLM
  chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.1,
                      streaming=True)

  # Create a prompt template for QA RAG System
  qa_template = """
                Use only the following pieces of context to answer the question at the end.
                If you don't know the answer, just say that you don't know,
                don't try to make up an answer. Keep the answer as concise as possible.

                {context}

                Question: {question}
                """
  qa_prompt = ChatPromptTemplate.from_template(qa_template)

  # This function formats retrieved documents before sending to LLM
  def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

  # Create a QA RAG System Chain
  qa_rag_chain = (
    {
      "context": itemgetter("question") # based on the user question get context docs
        |
      retriever
        |
      format_docs,
      "question": itemgetter("question") # user question
    }
      |
    qa_prompt # prompt with above user question and context
      |
    chatgpt # above prompt is sent to the LLM for response
      |
    StrOutputParser() # to parse the output to show on UI
  )
  # Set session variables to be accessed when user enters prompts in the app
  cl.user_session.set("qa_rag_chain", qa_rag_chain)


@cl.on_message
# this function is called whenever the user sends a prompt message in the app
async def on_user_message(message: cl.Message):

  # get the chain and memory objects from the session variables
  qa_rag_chain = cl.user_session.get("qa_rag_chain")

  # this will store the response from ChatGPT LLM
  chatgpt_message = cl.Message(content="")

  #Callback handler for handling the retriever and LLM processes.
  # Used to post the sources of the retrieved documents as a Chainlit element.
  class PostMessageHandler(BaseCallbackHandler):
    def __init__(self, msg: cl.Message):
      BaseCallbackHandler.__init__(self)
      self.msg = msg
      self.sources = []

    def on_retriever_end(self, documents, *, run_id, parent_run_id, **kwargs):
      source_ids = []
      for d in documents: # retrieved documents from retriever based on user query
        metadata = {
          "source": d.metadata["source"],
          "page": d.metadata["page"],
          "content": d.page_content[:200]
        }
        idx = (metadata["source"], metadata["page"])
        if idx not in source_ids: # store unique source documents
          source_ids.append(idx)
          self.sources.append(metadata)

    def on_llm_end(self, response, *, run_id, parent_run_id, **kwargs):
      if len(self.sources):
          sources_table = pd.DataFrame(self.sources[:3]).to_markdown()
          self.msg.elements.append(
            cl.Text(name="Sources", content=sources_table, display="inline")
          )

  # Stream the response from ChatGPT and show in real-time
  async with cl.Step(type="run", name="QA Assistant"):
    async for chunk in qa_rag_chain.astream(
        {"question": message.content},
        config=RunnableConfig(callbacks=[
            cl.LangchainCallbackHandler(),
            PostMessageHandler(chatgpt_message)
        ]),
    ):
        await chatgpt_message.stream_token(chunk)
  await chatgpt_message.send()

Overwriting app.py


## Start the app

In [8]:
!chainlit run app.py --port=8989 --watch &>./logs.txt &

## Change the Initial app screen

In [9]:
%%writefile chainlit.md

# Welcome to File QA RAG Chatbot 🤖

Please ask your question?

Writing chainlit.md


In [10]:
from pyngrok import ngrok
import yaml

# Terminate open tunnels if exist
ngrok.kill()

# Setting the authtoken
# Get your authtoken from `ngrok_credentials.yml` file
# with open('ngrok_credentials.yml', 'r') as file:
#     NGROK_AUTH_TOKEN = yaml.safe_load(file)
ngrok.set_auth_token(api_creds['NGORK_AUTH_TOKEN'])

# Open an HTTPs tunnel on port XXXX which you get from your `logs.txt` file
ngrok_tunnel = ngrok.connect(8989)
print("Chainlit App:", ngrok_tunnel.public_url)

Chainlit App: https://70a1-34-81-18-154.ngrok-free.app


## Remove running app processes

In [11]:
ngrok.kill()

In [12]:
!ps -ef | grep app

root           7       1  0 07:30 ?        00:00:11 /tools/node/bin/node /datalab/web/app.js
root        6651       1  2 07:57 ?        00:00:22 /usr/bin/python3 /usr/local/bin/chainlit run app
root       10432    1602  0 08:12 ?        00:00:00 /bin/bash -c ps -ef | grep app
root       10434   10432  0 08:12 ?        00:00:00 grep app


In [13]:
!sudo kill -9 6651